# ☁️ Chroma Cloud: The Official Online Vector Database

Just like **MongoDB Atlas** is the managed cloud service for MongoDB, **Chroma Cloud** (`cloud.trychroma.com`) is the official hosted SaaS for ChromaDB.

Instead of running a local server on your machine, your collections, documents, and vector indexes are stored and managed in the cloud. You can inspect your data directly inside your web browser on the **Chroma Cloud Web Console**.

### 🎯 Learning Goals:
1. **Load Cloud Credentials** securely from `.env` using `python-dotenv`.
2. **Connect to Chroma Cloud** using `chromadb.CloudClient()`.
3. **Create an Online Collection & Ingest Chunks** directly to the cloud.
4. **Query Your Cloud Vectors** across the internet.
5. **Inspect in Google Chrome**: View your collections and documents in the official Chroma Cloud web dashboard.

---
## Step 1: Load Environment Variables from `.env`

We keep secret API keys out of code by reading them from our `.env` file.
Ensure you have pasted your **`CHROMA_API_KEY`**, **`CHROMA_TENANT`**, and **`CHROMA_DATABASE`** into `.env`.

In [ ]:
import os
from dotenv import load_dotenv

# Load variables from .env file
load_dotenv(override=True)

api_key = os.getenv("CHROMA_API_KEY")
tenant = os.getenv("CHROMA_TENANT", "default_tenant")
database = os.getenv("CHROMA_DATABASE", "default_database")

if not api_key or api_key == "your_api_key_here":
    print("⚠️ Please add your valid CHROMA_API_KEY in the .env file before running!")
else:
    masked_key = api_key[:6] + "..." + api_key[-4:] if len(api_key) > 10 else "***"
    print(f"🔑 Loaded API Key: {masked_key}")
    print(f"🏢 Tenant:         {tenant}")
    print(f"🗄️ Database:       {database}")

---
## Step 2: Connect to Chroma Cloud via `CloudClient`

Chroma provides a built-in `chromadb.CloudClient()` that automatically connects to `api.trychroma.com` over HTTPS.

In [ ]:
import chromadb

# Initialize the Cloud Client
cloud_client = chromadb.CloudClient(
    api_key=api_key,
    tenant=tenant,
    database=database
)

# Ping server heartbeat
heartbeat = cloud_client.heartbeat()
print(f" Connected to Chroma Cloud successfully!")
print(f"💓 Cloud Heartbeat: {heartbeat} ns")

# List existing collections in the cloud database
existing_collections = cloud_client.list_collections()
print(f" Existing cloud collections ({len(existing_collections)}): {[c.name for c in existing_collections]}")

---
## Step 3: Ingest Data to Chroma Cloud

Let's create a cloud collection named **`cloud_knowledge_hub`** and push our structured text chunks to Chroma's cloud servers.

> 💡 **How vectorization works here**:
> The text chunks are vectorized by Chroma's local embedding engine on your machine and uploaded over TLS/HTTPS directly into your Chroma Cloud database.

In [ ]:
# Choose data source: company_knowledge_base.txt or personal_knowledge_dump_vector_db.txt
data_file = "personal_knowledge_dump_vector_db.txt" if os.path.exists("personal_knowledge_dump_vector_db.txt") else "company_knowledge_base.txt"

with open(data_file, "r", encoding="utf-8") as f:
    raw_text = f.read()

# Split into paragraphs / sections
paragraphs = [p.strip() for p in raw_text.split("\n\n") if len(p.strip()) > 80]

ids = [f"cloud_doc_{i+1:03d}" for i in range(len(paragraphs))]
metadatas = [{"source": data_file, "index": i+1} for i in range(len(paragraphs))]

print(f" Prepared {len(paragraphs)} chunks from '{data_file}' for cloud upload.")

# Create or retrieve cloud collection
cloud_col = cloud_client.get_or_create_collection(
    name="cloud_knowledge_hub",
    metadata={"environment": "production-cloud"}
)

# Upsert chunks to Chroma Cloud (in batches of 25 for fast upload)
batch_size = 25
for start_idx in range(0, len(paragraphs), batch_size):
    end_idx = start_idx + batch_size
    cloud_col.upsert(
        ids=ids[start_idx:end_idx],
        documents=paragraphs[start_idx:end_idx],
        metadatas=metadatas[start_idx:end_idx]
    )
    print(f"  Uploaded batch {start_idx+1} to {min(end_idx, len(paragraphs))}...")

print(f" Cloud collection '{cloud_col.name}' now holds {cloud_col.count()} documents!")

---
## Step 4: Run a Semantic Search Query on Chroma Cloud

Now we test searching against our cloud-hosted vector database over the internet:

In [ ]:
query = "What are the technical specs and hardware of the laptop?"

cloud_results = cloud_col.query(
    query_texts=[query],
    n_results=2
)

print(f"🔎 Query: '{query}'\n")
for i in range(len(cloud_results["ids"][0])):
    doc_id = cloud_results["ids"][0][i]
    dist = cloud_results["distances"][0][i]
    doc_text = cloud_results["documents"][0][i]
    
    print(f"Result #{i+1} [ID: {doc_id}, Distance: {dist:.4f}]:")
    print(f"{doc_text[:250]}...\n")

---
## 🌐 Step 5: View Your Data in Chrome (Chroma Cloud Web Console)

Now open Google Chrome and navigate to:
👉 **[https://cloud.trychroma.com](https://cloud.trychroma.com)**

1. Log into your account.
2. Click on your database (**`default_database`** or your chosen name).
3. You will see **`cloud_knowledge_hub`** listed with all uploaded documents, vector count, and metadata tags—just like MongoDB Atlas!